# Create a reproducible Chicago taxi-trip sample

This notebook creates an approximately 50 MiB CSV for testing the project without the 6.7 GB source file. Rows are sampled randomly across the complete observation period with a fixed seed. The sample is suitable for checking whether the workflow runs, but not for reproducing the report results because demand counts and taxi-day totals are lower than in the full dataset.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("../data")
SOURCE = DATA_DIR / "Taxi_Trips_(2024-)_20260502.csv"
OUTPUT = DATA_DIR / "Taxi_Trips_(2024-)_20260502_sample.csv"
SEED = 42
TARGET_BYTES = 50 * 1024**2

if not SOURCE.exists():
    raise FileNotFoundError(f"Source file not found: {SOURCE}")

# The 0.98 factor leaves a small buffer below the 50 MiB target.
sample_fraction = TARGET_BYTES / SOURCE.stat().st_size * 0.98
rng = np.random.default_rng(SEED)
OUTPUT.unlink(missing_ok=True)

total_rows = 0
sampled_rows = 0
write_header = True

for chunk in pd.read_csv(SOURCE, chunksize=100_000, low_memory=False):
    sampled_chunk = chunk.loc[rng.random(len(chunk)) < sample_fraction]
    total_rows += len(chunk)
    sampled_rows += len(sampled_chunk)

    if not sampled_chunk.empty:
        sampled_chunk.to_csv(OUTPUT, mode="a", header=write_header, index=False)
        write_header = False

print(f"Sampled {sampled_rows:,} of {total_rows:,} rows.")
print(f"Output size: {OUTPUT.stat().st_size / 1024**2:.2f} MiB")
print(f"Saved to: {OUTPUT.resolve()}")

Sampled 110,968 of 14,802,849 rows.
Output size: 47.36 MiB
Saved to: C:\Users\Georg\Documents\Studium\Master 2 Semester\AAA\AAA-assignment\data\Taxi_Trips_(2024-)_20260502_sample.csv


In [2]:
sample = pd.read_csv(OUTPUT, low_memory=False)
source_columns = pd.read_csv(SOURCE, nrows=0).columns.tolist()
sample_dates = pd.to_datetime(
    sample["Trip Start Timestamp"],
    format="%m/%d/%Y %I:%M:%S %p",
)

assert sample.columns.tolist() == source_columns
assert OUTPUT.stat().st_size <= TARGET_BYTES

print(f"Rows: {len(sample):,}")
print(f"Columns: {len(sample.columns)}")
print(f"Date range: {sample_dates.min()} to {sample_dates.max()}")
print(f"Calendar months represented: {sample_dates.dt.to_period('M').nunique()}")

Rows: 110,968
Columns: 23
Date range: 2024-01-01 00:00:00 to 2026-04-01 00:00:00
Calendar months represented: 28
